# Idealized PIG results

Choose a completed or finalized partial run below, then **Restart Kernel and Run All**.
This notebook uses NumPy, h5py and Matplotlib; it does not initialize MPI or Firedrake.
Histories stay on disk. Each plot reads a selected point or spatial snapshot rather than the full ensemble.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Locate the source even if the checkout directory has a different name.
source_hint = os.environ.get("ICESEE_SOURCE_ROOT")
candidates = ([Path(source_hint).expanduser()] if source_hint else []) + [Path.cwd(), *Path.cwd().parents]
for candidate in candidates:
    if (candidate / "src/postprocessing/pig.py").is_file():
        if "ICESEE" not in sys.modules:
            spec = importlib.util.spec_from_file_location(
                "ICESEE", candidate / "__init__.py", submodule_search_locations=[str(candidate)])
            package = importlib.util.module_from_spec(spec)
            sys.modules["ICESEE"] = package
            spec.loader.exec_module(package)
        break
from ICESEE.src.postprocessing.pig import PigResults, extract_datasets_from_h5

In [ ]:
results_dir = os.path.abspath(os.environ.get("ICESEE_DATA_PATH", "_modelrun_datasets"))
reference_dir = None  # Set only if shared truth/observations cannot be located from the run's params.yaml.
point_index = None    # Scalar-space DOF, not a packed state row. None selects an active-melt example point.
point_xy_m = None     # Alternatively supply (x, y) in metres when coordinates are available.
time_stride = 1      # 1 uses every saved timestep; larger values reduce reads for long histories.
snapshot_year = None # None selects the last available model time.
profile_year = None  # None selects the latest common written flowline profile.
show_members = True
figure_dir = None    # Optional directory for PNG exports; None writes no figures.
print(f"Reading ICESEE results from {results_dir}")

In [ ]:
from ICESEE.src.postprocessing.outputs import native_history_metadata
from ICESEE.src.postprocessing.pig import extract_datasets_from_h5

summary_path = os.path.join(results_dir, "true-wrong-icepack.h5")
datasets_tw = {}
if os.path.isfile(summary_path):
    datasets_tw = extract_datasets_from_h5(summary_path)
if "_ensemble_file" in globals():
    _ensemble_file.close()
ensemble_path = os.path.join(results_dir, "icesee_ensemble_data.h5")
if not os.path.isfile(ensemble_path):
    raise FileNotFoundError(
        f"{ensemble_path} is absent. Run the history finalizer after a failed "
        "native run; pruned checkpoints cannot be reconstructed."
    )
with h5py.File(ensemble_path, "r") as handle:
    native = handle.attrs.get("format") == "icesee-ensemble-history-v1"
    if not native and not handle.attrs.get("complete", True):
        raise ValueError("Ensemble history is incomplete")
if native:
    history_metadata = native_history_metadata(ensemble_path, summary=datasets_tw)
    t = history_metadata["t"]
    ind_m = history_metadata["obs_index"]
    observation_columns = history_metadata["obs_column_indices"]
    tm_m = history_metadata["obs_max_time"]
    run_mode = history_metadata["run_mode"]
    for message in history_metadata["warnings"]:
        print(message)
else:
    t = datasets_tw["t"]
    ind_m = datasets_tw["obs_index"]
    observation_columns = np.arange(len(ind_m))
    tm_m = datasets_tw["obs_max_time"][0]
    run_mode = datasets_tw["run_mode"][0]
_ensemble_file = h5py.File(ensemble_path, "r")
ensemble_vec_full = _ensemble_file["ensemble"]
ensemble_vec_mean = _ensemble_file["ensemble_mean"]
filter_type = "Analysis"

In [ ]:
if "results" in globals():
    results.close()
results = PigResults(results_dir, reference_dir=reference_dir)
ensemble_true_state = results.truth       # Lazy h5py.Dataset; slice rows/times explicitly.
ensemble_nurged_state = results.background # None if the background was not generated.
print("Ensemble (state, member, time):", results.ensemble.shape)
print("Saved time range:", results.t[0], "to", results.t[-1], "years")
print("Saved timestep sizes:", np.unique(np.round(np.diff(results.t), 12)))
print("Observed variables:", sorted(results.observed_variables) or "unknown; no observation markers drawn")
print("Background available:", results.background is not None)
print("Physical coordinates available:", results.coordinates is not None)
for warning in results.metadata.get("warnings", []):
    print(warning)

In [ ]:
h_indx = results.select_point(point_index, point_xy_m)
print("Selected scalar-space DOF:", h_indx)
if results.coordinates is not None:
    print("Physical location (x, y), metres:", results.coordinates[h_indx])
figures = {}

## State and forcing through time

Gray lines show individual members; the blue band is their 10–90% range, not a confidence interval.
Surface elevation (`s`, metres) is distinct from surface mass balance. Observation markers are drawn only for variables declared observed in the saved configuration.
A single DOF is a local diagnostic: examine several physical locations and flowlines before drawing conclusions about the whole ice sheet.

In [ ]:
figures["point_series"] = results.plot_point(h_indx, stride=time_stride, show_members=show_members)
plt.show()

In [ ]:
figures["point_error_spread"] = results.plot_point_diagnostics(h_indx, stride=time_stride)
plt.show()

## Flowline geometry and member profiles

These curves use the saved physical flowline, bed, written-step flags and actual profile times.
The chosen year is the closest common written profile to `profile_year`; it need not equal the final simulation time.

In [ ]:
try:
    figures["flowline"] = results.plot_profiles(profile_year)
    plt.show()
except (FileNotFoundError, ValueError) as error:
    print("Flowline view unavailable:", error)

## Spatial snapshots

Maps use verified scalar-space coordinates in the same reference ordering as the aggregate history.
They show values at mesh nodes without inventing rectangular cells or filling holes in the domain.
If `dof_coordinates.h5` is absent, the view falls back to an explicitly labelled DOF-index plot.
Background panels are omitted when no background was generated. Speed snapshots use the magnitude of the mean velocity vector; point time-series average the individual member speeds.

In [ ]:
figures["thickness_speed"] = results.plot_snapshot(snapshot_year, variables=("h", "speed"))
plt.show()

In [ ]:
figures["surface_melt"] = results.plot_snapshot(snapshot_year, variables=("s", "basal_melt_field"))
plt.show()

## Direct access for additional plots

`ensemble_vec_full` and `ensemble_vec_mean` remain HDF5 datasets for existing plotting routines.
Use `results.point("h", h_indx)` for bounded member/time arrays, or `results.snapshot("s", year=20)` for one spatial field.
Avoid `[:]` on complete truth/ensemble histories and do not import the simulation configuration bootstrap for plotting.
`extract_datasets_from_h5()` is available here for small metadata files only; it refuses large histories before loading them.

In [ ]:
point_data = results.point("h", h_indx, stride=time_stride)
print("Thickness members (selected times, member):", point_data["members"].shape)
print("Variables:", list(results.blocks))

## Optional figure export

Set `figure_dir` in the configuration cell to save the displayed figures. Changing `results_dir` requires rerunning from the configuration cell; reload closes the previous file handles.

In [ ]:
if figure_dir is not None:
    target = Path(figure_dir).expanduser()
    target.mkdir(parents=True, exist_ok=True)
    for name, figure in figures.items():
        figure.savefig(target / f"{name}.png", dpi=160, bbox_inches="tight")
    print("Figures saved to", target.resolve())

## File handles

The reader stays open for interactive plotting. Run the optional closing lines below when finished, or restart the kernel.
For a finalized partial run only committed times are available. Plots do not establish scientific validity by themselves.

In [ ]:
# results.close()
# _ensemble_file.close()